# Подготовка окружения

In [ ]:
# @title Установка библиотеки

!pip install git+https://github.com/iu5git/LiDARSegmentation@jupyter

In [ ]:
# @title Подключение Google Drive

from google.colab import output, drive
drive.mount('/content/drive')

# Обработка
## Описание
---
Объектом разработки является программа определения координат деревьев и их характеристик с использованием данных LiDAR.

Этапы при выполнении задачи автоматизированной таксации леса:
1. Обнаружение и фиксация местоположения дерева.
1. Сегментация отдельных деревьев.
1. Расчет параметров и характеристик дерева.

Разработанная программа позволит обнаружить деревья по результатам лазерного сканирования наземного базирования на участке лесного массива. pcPCD позволяет обрабатывать загружаемые в качестве входных данных файлы плотных облаков точек участка леса с целью определения местоположения и количества деревьев на участке.

Программа предлагает надежное обнаружение стволов деревьев, а методы искусственного интеллекта позволяют с высокой точностью обнаружить деревья, которые находятся в паспорте объекта. Позволяет извлечь файлы отдельных деревьев, определить их видовой состав, рассчитать таксационные параметры.


### Функции для запуска

In [ ]:
import sys
from pathlib import Path
import ast
from typing import Optional

sys.path.append('./lidar')

from lidarsegmentation.settings.coord_settings import CS
from lidarsegmentation.settings.seg_settings import SS
from lidarsegmentation.coordinates import coordinates
from lidarsegmentation.merge_coordinates import merge_coordinates
from lidarsegmentation.clear_excess_stumps import clear_excess_stumps
from lidarsegmentation.segmentation_vor import segmentation_vor as segmentation_vor_fn
from lidarsegmentation.segmentation_ram import segmentation_ram as segmentation_ram_fn
from lidarsegmentation.segmentation_clear import segmentation_clear as segmentation_clear_fn
from lidarsegmentation.predict import predict
from lidarsegmentation.parameters import parameters

def load_coordinate_settings():
    yaml_path = Path(settings_file_path.strip()) if settings_file_path.strip() else None
    if yaml_path and yaml_path.is_file():
        cs = CS()
        cs.set(str(yaml_path))
        return cs

    project_dir = Path(project_folder.strip()) if project_folder.strip() else None
    points_path = Path(point_cloud_file.strip()) if point_cloud_file.strip() else None
    traj_path   = Path(person_traj_file.strip()) if person_traj_file.strip() else None
    shape_path  = Path(shape_file.strip())       if shape_file.strip()       else None

    cs = CS(
        FLAG_cut_data   = FLAG_cut_data,
        FLAG_make_cells = FLAG_make_cells,
        FLAG_make_stumps= FLAG_make_stumps,
        cut_data_method = cut_data_method,
        LOW             = float(LOW),
        UP              = float(UP),
        x_shift         = float(x_shift),
        y_shift         = float(y_shift),
        z_shift         = float(z_shift),
        algo            = algo,
        n_clusters      = int(n_clusters),
        cell_size       = float(cell_size),
        height_limit_1  = float(height_limit_1),
        height_limit_2  = float(height_limit_2),
        eps_XY          = float(eps_XY),
        eps_Z           = float(eps_Z),
        path_base       = str(project_dir) if project_dir else "",
        fname_points    = str(points_path) if points_path else "",
        fname_traj      = str(traj_path)   if traj_path   else "",
        fname_shape     = str(shape_path)  if shape_path  else "",
    )
    return cs

def load_segmentation_settings():
    yaml_path = Path(settings_file_path.strip()) if settings_file_path.strip() else None
    if yaml_path and yaml_path.is_file():
        ss = SS()
        ss.set(str(yaml_path))
        return ss

    zt = ast.literal_eval(z_thresholds)
    es = ast.literal_eval(eps_steps)
    mp = ast.literal_eval(min_pts)

    project_dir = Path(project_folder.strip()) if project_folder.strip() else None
    points_path = Path(point_cloud_file.strip()) if point_cloud_file.strip() else None
    shape_path  = Path(shape_file.strip())       if shape_file.strip()       else None
    coord_path  = Path(coordinates_file.strip()) if coordinates_file.strip() else None

    if (segmentation_voronoi or segmentation_ram or segmentation_clear
        or predict_labels or estimate_parameters):
        if not coord_path and points_path and points_path.suffix:
            sname = points_path.stem
            coord_path = project_dir / f"{sname}_Clear_Excess.csv" if project_dir else Path(f"{sname}_Clear_Excess.csv")

    ss = SS(
        path_base      = str(project_dir) if project_dir else "",
        fname_points   = str(points_path) if points_path else "",
        fname_shape    = str(shape_path)  if shape_path  else "",
        csv_name_coord = str(coord_path)  if coord_path  else "",
        first_num      = int(first_num),
        STEP           = float(STEP),
        z_thresholds   = zt,
        eps_steps      = es,
        min_pts        = mp,
    )
    return ss

def run_coordinates_pipeline(cs: CS):
    try:
        if coordinates_7000:
            coordinates(7000, cs)
            print("Done processing Coordinates(int = 7000)")
        if coordinates_5000:
            coordinates(5000, cs)
            print("Done processing Coordinates(int = 5000)")
        if coordinates_1000:
            coordinates(1000, cs)
            print("Done processing Coordinates(int = 1000)")
        if should_merge_coordinates:
            merge_coordinates(cs)
            print("Done processing Merge Coordinates")
        if should_clear_excess_stumps:
            clear_excess_stumps(cs)
            print("Done processing Clear Excess Stumps")
    except Exception as e:
        print("[ERROR in run_coordinates_pipeline]:", e)
        raise e

def run_segmentation_pipeline(ss: SS):
    try:
        if segmentation_voronoi:
            segmentation_vor_fn(ss, make_binding=True)
            print("Done processing Segmentation Voronoi")
        if segmentation_ram:
            segmentation_ram_fn(ss)
            print("Done processing Segmentation RAM")
        if segmentation_clear:
            segmentation_clear_fn(ss)
            print("Done processing Segmentation Clear")
    except Exception as e:
        print("[ERROR in run_segmentation_pipeline]:", e)
        raise e

def run_other_pipeline(ss: SS):
    try:
        if predict_labels:
            model_name = 'cpl1-1024-rp-s1024-pn2'
            path_file = Path(ss.path_base) / ss.step1_folder_name / ss.step2_folder_name / ss.step3_folder_name
            predict(str(path_file), model_name)
            print("Done processing Predict Labels")
        if estimate_parameters:
            path_file = Path(ss.path_base) / ss.step1_folder_name / ss.step2_folder_name / ss.step3_folder_name
            parameters(ss, str(path_file))
            print("Done processing Estimate Parameters")
    except Exception as e:
        print("[ERROR in run_other_pipeline]:", e)

def start_colab_coordinates():
    cs = load_coordinate_settings()
    run_coordinates_pipeline(cs)
    print("Coordinates step done.")

def start_colab_segmentation():
    ss = load_segmentation_settings()
    run_segmentation_pipeline(ss)
    run_other_pipeline(ss)
    print("Segmentation and other steps done.")

### Конфигурация и запуск программы

In [ ]:
# @markdown # Конфигурация


# @markdown ---

# @markdown Параметры можно указать в .yaml файле для воспроизводимости результатов.
# @markdown В этом случае, все остальные параметры будут проигнорированы.

# @markdown Файл настроек (*.yaml)
settings_file_path = ""    # @param {type:"string", placeholder:"Путь до .yaml"}


# @markdown В противном случае, укажите все неопциональные параметры ниже.

# @markdown Папка проекта **(*oбязательно)**
project_folder = "/content/drive/MyDrive/lidar/new_test/"        # @param {type:"string"}
# @markdown Файл с данными облака (\*.las, \*.pcd) **(*oбязательно)**
point_cloud_file = "/content/drive/MyDrive/lidar/01_04_05_oct9.las"      # @param {type:"string"}
# @markdown Файл с треком человека (\*.las, \*.pcd)
person_traj_file = ""      # @param {type:"string"}
# @markdown Файл с границами участка (\*.shp)
shape_file = ""            # @param {type:"string"}

In [ ]:
# @markdown # **Поиск координат**
# @markdown ---

# @markdown Coordinates (int = 7000)
coordinates_7000 = True  # @param {type:"boolean"}
# @markdown Coordinates (int = 5000)
coordinates_5000 = True  # @param {type:"boolean"}
# @markdown Coordinates (int = 1000)
coordinates_1000 = True  # @param {type:"boolean"}
# @markdown Merge Coordinates
should_merge_coordinates = True # @param {type:"boolean"}
# @markdown Clear Excess Stumps
should_clear_excess_stumps = True # @param {type:"boolean"}


# @markdown

# @markdown ## **Настройки поиска координат**
# @markdown ---

# @markdown Обрезка данных по высоте и границам участка
FLAG_cut_data = True  # @param {type:"boolean"}
# @markdown Выделение подобластей
FLAG_make_cells = True # @param {type:"boolean"}
# @markdown Выделение пеньков деревьев и вычисление координат
FLAG_make_stumps = True # @param {type:"boolean"}
# @markdown Метод выделения подобластей
cut_data_method = "voronoi_tessellation"  # @param ["voronoi_tessellation","flood_fill"]
# @markdown Нижняя граница рассматриваемого слоя точек
LOW = 0.0  # @param {type:"number"}
# @markdown Верхняя граница рассматриваемого слоя точек
UP = 3.0   # @param {type:"number"}
# @markdown Сдвиг по X облака точек
x_shift = 0.0  # @param {type:"number"}
# @markdown Сдвиг по Y облака точек
y_shift = 0.0  # @param {type:"number"}
# @markdown Сдвиг по Z облака точек
z_shift = 0.0  # @param {type:"number"}
# @markdown Метод кластеризации
algo = "birch"  # @param ["birch","spectral","kmeans"]
# @markdown Количество кластеров
n_clusters = 3     # @param {type:"integer"}
# @markdown Размерность ячейки для выделения ячеек по границам трека
cell_size = 0.20     # @param {type:"number"}
# @markdown Лимит по минимальной высоте извлеченных пеньков (первый этап)
height_limit_1 = 1.25  # @param {type:"number"}
# @markdown Лимит по минимальной высоте извлеченных пеньков (второй этап)
height_limit_2 = 1.35  # @param {type:"number"}
# @markdown Параметр алгоритма DBSCAN по осям XY
eps_XY = 0.08          # @param {type:"number"}
# @markdown Параметр алгоритма DBSCAN по оси Z
eps_Z = 0.7            # @param {type:"number"}


start_colab_coordinates()

In [ ]:
# @markdown # **Сегментация**
# @markdown ---

# @markdown Segmentation Voronoi
segmentation_voronoi = False  # @param {type:"boolean"}
# @markdown Segmentation RAM
segmentation_ram = True      # @param {type:"boolean"}
# @markdown Segmentation Clear
segmentation_clear = False    # @param {type:"boolean"}
# @markdown Predict Labels
predict_labels = False        # @param {type:"boolean"}
# @markdown Estimate Parameters
estimate_parameters = False   # @param {type:"boolean"}


# @markdown

# @markdown ## **Параметры сегментации**
# @markdown ---

# @markdown Файл с координатами (\*.csv) (*oбязательно)
coordinates_file = "/content/drive/MyDrive/lidar/01_04_05_oct9_Clear_Excess.csv"      # @param {type:"string"}
# @markdown Номер дерева, с которого начнется извлечение
first_num = 0  # @param {type:"integer"}
# @markdown Шаг просмотра по высоте
STEP = 2.5     # @param {type:"number"}
# @markdown Пороги по высоте, до которых действуют `eps_steps` и `min_pts`
z_thresholds = "[0.5, 0.625, 0.695, 0.75, 0.875, 1]"  # @param {type:"string"}
# @markdown `eps` алгоритма DBSCAN (добавляется к 0.35) при `z_thresholds`
eps_steps    = "[0.01, 0.15, 0.35, 0.5, 0.6, 0.7]"    # @param {type:"string"}
# @markdown `minPts` алгоритма DBSCAN при `z_thresholds`
min_pts      = "[50, 50, 50, 50, 45, 40]"            # @param {type:"string"}



start_colab_segmentation()